# LIBRARIES

In [ ]:
!pip install -q optuna torchinfo torchmetrics

In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import os
import random

from xgboost.sklearn import XGBClassifier
from xgboost import XGBClassifier  
from lightgbm.sklearn import LGBMClassifier
from sklearn.metrics import f1_score

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold

from functools import partial

import optuna

In [2]:
## Pytorch Import
import torch
import torch.nn as nn

from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader

import torchmetrics
from torchmetrics.classification import BinaryF1Score

In [3]:
import re
import gc
import time
import string
import psutil

import copy
from copy import deepcopy

# Utils
from tqdm.auto import tqdm, trange

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

In [4]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)

In [5]:
!unzip -p /kaggle/input/bosch-production-line-performance/train_numeric.csv.zip | wc -l

1183748


In [6]:
!unzip -p /kaggle/input/bosch-production-line-performance/train_categorical.csv.zip | wc -l

1183748


In [7]:
!unzip -p /kaggle/input/bosch-production-line-performance/train_date.csv.zip | wc -l

1183748


# EDA

In [8]:
def reduce_numeric_memory(df):
    """Sayısal verilerde otomatik tip küçültme"""
    start_mem = df.memory_usage().sum() / 1024**2
    print(f"Başlangıç hafıza kullanımı: {start_mem:.2f} MB")
    
    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object and str(col_type)[:3] != 'dat':
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type).startswith('int'):
                if c_min >= 0:
                    if c_max < 255:
                        df[col] = df[col].astype('uint8')
                    elif c_max < 65535:
                        df[col] = df[col].astype('uint16')
                    elif c_max < 4294967295:
                        df[col] = df[col].astype('uint32')
                    else:
                        df[col] = df[col].astype('uint64')
                else:
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype('int8')
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype('int16')
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype('int32')
                    else:
                        df[col] = df[col].astype('int64')

            elif str(col_type).startswith('float'):
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype('float16')
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype('float32')
                else:
                    df[col] = df[col].astype('float64')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Sonraki hafıza kullanımı: {end_mem:.2f} MB  Azalma: {100 * (start_mem - end_mem) / start_mem:.1f}%")
    return df


In [9]:
def reduce_categorical_memory(df):
    """Kategorik verilerde tip küçültme (object → category)"""
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Başlangıç hafıza kullanımı: {start_mem:.2f} MB")

    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Sonraki hafıza kullanımı: {end_mem:.2f} MB  Azalma: {100 * (start_mem - end_mem) / start_mem:.1f}%")
    return df

In [10]:
import zipfile
from typing import List

def takeDataset_chunks(path:str) -> pd.DataFrame:
    response_list = []

    try:
        reader = pd.read_csv(path, compression='zip', chunksize=100000)
    
        for chunk in reader:
            if 'Response' in chunk.columns:
                filtered = chunk[chunk['Response'] == 1]
            response_list.append(filtered)

            del chunk, filtered
            gc.collect()

        df_response = pd.concat(response_list, axis=0).reset_index(drop=True)
        print(f"Toplam Response=1 örnek: {df_response.shape[0]}")
        return df_response
    except Exception as e:
        print(f"Error readinf {path} : {e} ")
        return pd.DataFrame()

In [11]:
train_paths = {
    'categorical': '/kaggle/input/bosch-production-line-performance/train_categorical.csv.zip',
    'date': '/kaggle/input/bosch-production-line-performance/train_date.csv.zip',
    'numeric': '/kaggle/input/bosch-production-line-performance/train_numeric.csv.zip'
}

In [12]:
df_numeric = takeDataset_chunks(train_paths['numeric'])
df_numeric_rnm = reduce_numeric_memory(df_numeric)

Toplam Response=1 örnek: 6879
Başlangıç hafıza kullanımı: 50.91 MB
Sonraki hafıza kullanımı: 12.73 MB  Azalma: 75.0%


In [13]:
response_num_ids = df_numeric_rnm['Id'].values

#### get matching rows 

In [14]:
def get_matching_rows(path: str, id_list: List[int]) -> pd.DataFrame:
    match_list = []
    reader = pd.read_csv(path, compression='zip', chunksize=100000)

    for chunk in reader:
        chunk = chunk[chunk['Id'].isin(id_list)]
        match_list.append(chunk)

        del chunk
        gc.collect()

    df_matched = pd.concat(match_list, axis=0).reset_index(drop=True)
    print(f"{path.split('/')[-1]}: {df_matched.shape[0]} eşleşen örnek")
    return df_matched 

In [15]:
df_categorical = get_matching_rows(train_paths['categorical'], response_num_ids)
df_categorical_rcm = reduce_categorical_memory(df_categorical)

train_categorical.csv.zip: 6879 eşleşen örnek
Başlangıç hafıza kullanımı: 434.55 MB
Sonraki hafıza kullanımı: 21.83 MB  Azalma: 95.0%


In [16]:
df_date = get_matching_rows(train_paths['date'], response_num_ids)
df_date_rnm = reduce_numeric_memory(df_date)

train_date.csv.zip: 6879 eşleşen örnek
Başlangıç hafıza kullanımı: 60.72 MB
Sonraki hafıza kullanımı: 15.63 MB  Azalma: 74.3%


In [17]:
del df_categorical
del df_date
del df_numeric
gc.collect()

0

#### Merge by Id


In [18]:
df_train_full = df_numeric_rnm.merge(df_categorical_rcm, on='Id', how='left')
df_train_full = df_train_full.merge(df_date_rnm, on='Id', how='left')

print("Final birleştirilmiş Response=1 veri:", df_train_full.shape)

Final birleştirilmiş Response=1 veri: (6879, 4266)


In [19]:
df_train_full.head()

,Id,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,...,L3_S50_D4246,L3_S50_D4248,L3_S50_D4250,L3_S50_D4252,L3_S50_D4254,L3_S51_D4255,L3_S51_D4257,L3_S51_D4259,L3_S51_D4261,L3_S51_D4263
0,1053,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1250,0.075012,0.101013,-0.178955,-0.215942,-0.013000,0.070007,-0.022003,-0.151978,0.086975,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1350,0.068970,0.040985,0.330078,0.330078,-0.099976,-0.293945,0.008003,0.088013,-0.091980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1793,0.003000,-0.026001,0.330078,0.293945,0.073975,0.161011,0.022003,0.128052,-0.198975,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2347,-0.114014,-0.161011,0.330078,0.330078,-0.013000,0.116028,0.045013,0.288086,0.036011,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df_train_full.tail()

,Id,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,...,L3_S50_D4246,L3_S50_D4248,L3_S50_D4250,L3_S50_D4252,L3_S50_D4254,L3_S51_D4255,L3_S51_D4257,L3_S51_D4259,L3_S51_D4261,L3_S51_D4263
6874,2366099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,555.5,555.5,555.5,555.5,555.5,555.5,555.5,555.5,555.5,555.5
6875,2366124,0.101013,0.048004,0.003000,0.003000,-0.099976,-0.248047,-0.014999,-0.072021,0.131958,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6876,2366209,0.016006,0.040985,-0.178955,-0.178955,0.073975,0.116028,-0.014999,-0.072021,0.029999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6877,2366505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6878,2366692,0.134033,0.122986,0.330078,0.330078,-0.013000,0.070007,-0.007000,0.008003,0.005001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df_train_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6879 entries, 0 to 6878
Columns: 4266 entries, Id to L3_S51_D4263
dtypes: category(1977), float16(2113), float64(174), uint32(1), uint8(1)
memory usage: 50.1 MB


In [22]:
df_train_full.describe()

,Id,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,...,L3_S50_D4246,L3_S50_D4248,L3_S50_D4250,L3_S50_D4252,L3_S50_D4254,L3_S51_D4255,L3_S51_D4257,L3_S51_D4259,L3_S51_D4261,L3_S51_D4263
count,6.879000e+03,3608.000000,3608.000000,3608.000000,3608.000000,3608.000000,3608.000000,3608.000000,3608.000000,3608.000000,...,158.000000,158.000000,158.000000,158.000000,158.000000,302.000000,302.000000,302.000000,302.000000,302.000000
mean,1.186405e+06,-0.000399,0.001390,-0.030899,-0.030624,0.003441,-0.004429,-0.001344,-0.008926,-0.007706,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
std,6.796131e+05,0.077637,0.090332,0.198853,0.199097,0.095825,0.163696,0.018372,0.098022,0.113831,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
min,1.053000e+03,-0.283936,-0.346924,-0.378906,-0.396973,-0.404053,-0.565918,-0.044006,-0.232056,-0.336914,...,1.480469,1.480469,1.480469,1.480469,1.480469,1.480469,1.480469,1.480469,1.480469,1.480469
25%,6.083060e+05,-0.049011,-0.056000,-0.178955,-0.178955,-0.056000,-0.112000,-0.014999,-0.072021,-0.086975,...,558.000000,558.000000,558.000000,558.000000,558.000000,553.500000,553.500000,553.500000,553.500000,553.500000
50%,1.175056e+06,0.003000,0.004002,-0.052002,-0.052002,0.031006,0.024994,-0.007000,-0.032013,-0.010002,...,1222.000000,1222.000000,1222.000000,1222.000000,1222.000000,839.000000,839.000000,839.000000,839.000000,839.000000
75%,1.775558e+06,0.049011,0.062988,0.003000,0.003000,0.073975,0.116028,0.008003,0.048004,0.065979,...,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000
max,2.366692e+06,0.258057,0.271973,0.566895,0.565918,0.291992,0.206055,0.081970,0.447998,0.393066,...,1457.000000,1457.000000,1457.000000,1457.000000,1457.000000,1457.000000,1457.000000,1457.000000,1457.000000,1457.000000


In [23]:
# NaN oranlarını hesapla
nan_info = df_train_full.isnull().mean().sort_values(ascending=False)

# Sadece NaN içerenleri al
nan_info = nan_info[nan_info > 0]

# Her bir sütun için oran ve toplam sayıyı yazdır
for col in nan_info.index:
    ratio = nan_info[col]
    total_nan = df_train_full[col].isnull().sum()
    print(f"{col:<30} | NaN oranı: {ratio:.2%} | NaN sayısı: {total_nan}")

L3_S46_D4135                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4033                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4037                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4041                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4045                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4049                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4053                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4057                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S42_D4029                   | NaN oranı: 100.00% | NaN sayısı: 6879
L1_S24_F1054                   | NaN oranı: 100.00% | NaN sayısı: 6879
L1_S24_F1052                   | NaN oranı: 100.00% | NaN sayısı: 6879
L1_S24_F1050                   | NaN oranı: 100.00% | NaN sayısı: 6879
L1_S24_D1562                   | NaN oranı: 100.00% | NaN sayısı: 6879
L1_S24_D1158                   | NaN oranı: 100.00% | NaN sayısı: 6879
L3_S47

#### %55'ten fazla NaN olanları silelim


In [24]:
cols_to_drop = nan_info[nan_info > 0.55].index
df_train_clean = df_train_full.drop(columns=cols_to_drop)

print(f"Silinen sütun sayısı: {len(cols_to_drop)}")
print(f"Kalan sütun sayısı: {df_train_clean.shape[1]}")

Silinen sütun sayısı: 3843
Kalan sütun sayısı: 423


In [25]:
df_train_clean.shape

(6879, 423)

### Lets fill nan values 

In [26]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

def debug_and_fix_imputation(df: pd.DataFrame, knn_neighbors=3, verbose=True) -> pd.DataFrame:
    """
    Advanced imputation with debugging information
    """
    df_new = df.copy()
    
    if verbose:
        print(f"Initial shape: {df_new.shape}")
        print(f"Initial NaNs: {df_new.isnull().sum().sum()}")
        print(f"Data types: {df_new.dtypes.value_counts()}")
    
    # --- Debug: Check for infinite values ---
    inf_cols = []
    for col in df_new.columns:
        if df_new[col].dtype in ['float64', 'int64']:
            if np.isinf(df_new[col]).any():
                inf_cols.append(col)
    
    if inf_cols and verbose:
        print(f"Columns with infinite values: {inf_cols}")
    
    # Replace infinite values with NaN
    df_new = df_new.replace([np.inf, -np.inf], np.nan)
    
    # --- Date Columns ---
    date_cols = [col for col in df_new.columns if '_D' in col]
    if verbose:
        print(f"Date columns found: {len(date_cols)}")
    
    for col in date_cols:
        # Convert to numeric if not already
        if not pd.api.types.is_numeric_dtype(df_new[col]):
            df_new[col] = pd.to_numeric(df_new[col], errors='coerce')
        
        # Check for valid range
        col_min = df_new[col].min()
        col_max = df_new[col].max()
        
        if pd.isna(col_min) or pd.isna(col_max):
            # All values are NaN
            df_new[col] = 0.0
        elif col_max - col_min == 0:
            # All non-NaN values are the same
            df_new[col] = df_new[col].fillna(0.0)
            df_new[col] = 0.0
        else:
            # Normalize (0-1)
            df_new[col] = (df_new[col] - col_min) / (col_max - col_min)
            # Fill missing values with mean
            df_new[col] = df_new[col].fillna(df_new[col].mean())
    
    if verbose:
        print(f"After date processing NaNs: {df_new.isnull().sum().sum()}")
    
    # --- Categorical Columns ---
    cat_cols = df_new.select_dtypes(include=['object', 'category']).columns.tolist()
    if verbose:
        print(f"Categorical columns found: {len(cat_cols)}")
    
    for col in cat_cols:
        # Convert to string
        df_new[col] = df_new[col].astype(str)
        
        # Replace 'nan' string with actual NaN, then fill with "unknown"
        df_new[col] = df_new[col].replace('nan', np.nan)
        df_new[col] = df_new[col].fillna("unknown")
        
        # Frequency Encoding
        freq = df_new[col].value_counts(normalize=True)
        df_new[col] = df_new[col].map(freq)
        
        # Fill any remaining NaNs with 0.0
        df_new[col] = df_new[col].fillna(0.0)
    
    if verbose:
        print(f"After categorical processing NaNs: {df_new.isnull().sum().sum()}")
    
    # --- Numerical Columns ---
    # Get all numeric columns
    all_numeric_cols = df_new.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    if verbose:
        print(f"Numeric columns for KNN: {len(all_numeric_cols)}")
        nan_per_col = df_new[all_numeric_cols].isnull().sum()
        cols_with_nans = nan_per_col[nan_per_col > 0]
        print(f"Columns still with NaNs: {len(cols_with_nans)}")
        if len(cols_with_nans) > 0:
            print("Top 10 columns with most NaNs:")
            print(cols_with_nans.sort_values(ascending=False).head(10))
    
    # Check if we have enough non-NaN rows for KNN
    non_nan_rows = df_new[all_numeric_cols].dropna().shape[0]
    if verbose:
        print(f"Rows without any NaN: {non_nan_rows}")
    
    if non_nan_rows < knn_neighbors:
        if verbose:
            print(f"Not enough complete rows for KNN (need at least {knn_neighbors})")
            print("Falling back to mean imputation...")
        
        # Fall back to mean imputation
        for col in all_numeric_cols:
            if df_new[col].isnull().any():
                mean_val = df_new[col].mean()
                if pd.isna(mean_val):
                    # If mean is also NaN (all values are NaN), fill with 0
                    df_new[col] = df_new[col].fillna(0.0)
                else:
                    df_new[col] = df_new[col].fillna(mean_val)
    else:
        # Apply KNN imputation
        try:
            knn = KNNImputer(n_neighbors=min(knn_neighbors, non_nan_rows))
            df_new[all_numeric_cols] = knn.fit_transform(df_new[all_numeric_cols])
        except Exception as e:
            if verbose:
                print(f"KNN imputation failed: {e}")
                print("Falling back to mean imputation...")
            
            # Fall back to mean imputation
            for col in all_numeric_cols:
                if df_new[col].isnull().any():
                    mean_val = df_new[col].mean()
                    if pd.isna(mean_val):
                        df_new[col] = df_new[col].fillna(0.0)
                    else:
                        df_new[col] = df_new[col].fillna(mean_val)
    
    if verbose:
        print(f"Final shape: {df_new.shape}")
        print(f"Final NaNs: {df_new.isnull().sum().sum()}")
    
    return df_new

# Also create a simpler version that handles edge cases better
def robust_impute_and_encode(df: pd.DataFrame, knn_neighbors=3) -> pd.DataFrame:
    """
    Robust version with better error handling
    """
    df_new = df.copy()
    
    # Replace infinite values
    df_new = df_new.replace([np.inf, -np.inf], np.nan)
    
    # --- Date Columns ---
    date_cols = [col for col in df_new.columns if '_D' in col]
    for col in date_cols:
        if not pd.api.types.is_numeric_dtype(df_new[col]):
            df_new[col] = pd.to_numeric(df_new[col], errors='coerce')
        
        # Normalize only if we have valid range
        col_min = df_new[col].min()
        col_max = df_new[col].max()
        
        if pd.notna(col_min) and pd.notna(col_max) and col_max != col_min:
            df_new[col] = (df_new[col] - col_min) / (col_max - col_min)
            df_new[col] = df_new[col].fillna(df_new[col].mean())
        else:
            df_new[col] = df_new[col].fillna(0.0)
    
    # --- Categorical Columns ---
    cat_cols = df_new.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in cat_cols:
        df_new[col] = df_new[col].astype(str).replace('nan', np.nan)
        df_new[col] = df_new[col].fillna("unknown")
        
        freq = df_new[col].value_counts(normalize=True)
        df_new[col] = df_new[col].map(freq).fillna(0.0)
    
    # --- Final imputation for all remaining NaNs ---
    # Simple approach: fill remaining NaNs with median/mode/0
    for col in df_new.columns:
        if df_new[col].isnull().any():
            if df_new[col].dtype in ['float64', 'int64']:
                # Use median for numeric
                median_val = df_new[col].median()
                df_new[col] = df_new[col].fillna(median_val if pd.notna(median_val) else 0.0)
            else:
                # Use mode for non-numeric (shouldn't happen after processing above)
                mode_val = df_new[col].mode()
                fill_val = mode_val[0] if len(mode_val) > 0 else "unknown"
                df_new[col] = df_new[col].fillna(fill_val)
    
    return df_new

In [34]:
# Usage examples:
# df_processed = debug_and_fix_imputation(df_train_clean, verbose=True)
# or
df_processed = robust_impute_and_encode(df_train_clean)

In [27]:
df_processed = robust_impute_and_encode(df_train_clean)

In [28]:
print("Final shape:", df_processed.shape)
print("Kalan NaN:", df_processed.isnull().sum().sum())  # 0 olmalı

Final shape: (6879, 423)
Kalan NaN: 0


In [29]:
df_processed.head()

,Id,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,...,L3_S36_D3928,L3_S36_D3932,L3_S36_D3936,L3_S36_D3940,L3_S37_D3942,L3_S37_D3943,L3_S37_D3945,L3_S37_D3947,L3_S37_D3949,L3_S37_D3951
0,1053,0.003000,0.018997,-0.178955,-0.178955,0.031006,0.116028,-0.014999,-0.072021,-0.061005,...,0.175537,0.175537,0.175537,0.175537,0.175537,0.175537,0.175537,0.175537,0.175537,0.175537
1,1250,0.075012,0.101013,-0.178955,-0.215942,-0.013000,0.070007,-0.022003,-0.151978,0.086975,...,0.362061,0.362061,0.362061,0.362061,0.362061,0.362061,0.362061,0.362061,0.362061,0.362061
2,1350,0.068970,0.040985,0.330078,0.330078,-0.099976,-0.293945,0.008003,0.088013,-0.091980,...,0.380371,0.380371,0.380371,0.380371,0.380371,0.380371,0.380371,0.380371,0.380371,0.380371
3,1793,0.003000,-0.026001,0.330078,0.293945,0.073975,0.161011,0.022003,0.128052,-0.198975,...,0.671875,0.671875,0.671875,0.671875,0.671875,0.671875,0.671875,0.671875,0.671875,0.671875
4,2347,-0.114014,-0.161011,0.330078,0.330078,-0.013000,0.116028,0.045013,0.288086,0.036011,...,0.446045,0.446045,0.446045,0.446045,0.445801,0.445801,0.445801,0.445801,0.445801,0.445801


In [30]:
duplicates = df_processed.duplicated()
print(f"Aynı satırdan {duplicates.sum()} adet bulundu.")

Aynı satırdan 0 adet bulundu.


In [31]:
low_var_cols = df_processed.loc[:, df_processed.nunique() <= 1].columns
print(f"Tüm değerleri aynı olan {len(low_var_cols)} sütun var: {list(low_var_cols)}")
#df_processed = df_processed.drop(columns=low_var_cols)

Tüm değerleri aynı olan 12 sütun var: ['L3_S30_F3594', 'L3_S30_F3599', 'L3_S30_F3614', 'L3_S30_F3619', 'L3_S30_F3654', 'L3_S30_F3659', 'L3_S30_F3694', 'L3_S30_F3699', 'L3_S30_F3714', 'L3_S30_F3719', 'L3_S34_F3878', 'Response']


In [32]:
low_var_cols = df_processed.loc[:, df_processed.nunique() <= 1].columns
cols_to_drop = [col for col in low_var_cols if col != 'Response']
print(f"Silinecek sabit sütunlar: {len(cols_to_drop)} adet → {cols_to_drop}")
df_processed = df_processed.drop(columns=cols_to_drop)

Silinecek sabit sütunlar: 11 adet → ['L3_S30_F3594', 'L3_S30_F3599', 'L3_S30_F3614', 'L3_S30_F3619', 'L3_S30_F3654', 'L3_S30_F3659', 'L3_S30_F3694', 'L3_S30_F3699', 'L3_S30_F3714', 'L3_S30_F3719', 'L3_S34_F3878']


In [33]:
print("Yeni shape:", df_processed.shape)
print("Kalan tek tip sütun var mı?:", df_processed.loc[:, df_processed.nunique() <= 1].columns.tolist())

Yeni shape: (6879, 412)
Kalan tek tip sütun var mı?: ['Response']


# GENERATE DATA PART 

In [ ]:
!pip install sdv -q

In [40]:
from sdv.single_table import GaussianCopulaSynthesizer, TVAESynthesizer, CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

df_response_1 = df_processed[df_processed['Response'] == 1].copy()
df_response_1_for_gan = df_response_1.drop(columns=['Id', 'Response'])

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_response_1_for_gan)

In [41]:
print(metadata.to_dict())

for column in df_response_1_for_gan.columns:
    if df_response_1_for_gan[column].dtype in ['int64', 'float64']:
        metadata.update_column(column, sdtype='numerical')

{'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1', 'columns': {'L0_S0_F0': {'sdtype': 'numerical'}, 'L0_S0_F2': {'sdtype': 'numerical'}, 'L0_S0_F4': {'sdtype': 'numerical'}, 'L0_S0_F6': {'sdtype': 'numerical'}, 'L0_S0_F8': {'sdtype': 'numerical'}, 'L0_S0_F10': {'sdtype': 'numerical'}, 'L0_S0_F12': {'sdtype': 'numerical'}, 'L0_S0_F14': {'sdtype': 'numerical'}, 'L0_S0_F16': {'sdtype': 'numerical'}, 'L0_S0_F18': {'sdtype': 'numerical'}, 'L0_S0_F20': {'sdtype': 'numerical'}, 'L0_S0_F22': {'sdtype': 'numerical'}, 'L0_S1_F24': {'sdtype': 'numerical'}, 'L0_S1_F28': {'sdtype': 'numerical'}, 'L0_S8_F144': {'sdtype': 'numerical'}, 'L0_S8_F146': {'sdtype': 'numerical'}, 'L0_S8_F149': {'sdtype': 'numerical'}, 'L3_S29_F3315': {'sdtype': 'numerical'}, 'L3_S29_F3318': {'sdtype': 'numerical'}, 'L3_S29_F3321': {'sdtype': 'numerical'}, 'L3_S29_F3324': {'sdtype': 'numerical'}, 'L3_S29_F3327': {'sdtype': 'numerical'}, 'L3_S29_F3330': {'sdtype': 'numerical'}, 'L3_S29_F3333': {'sdtype': 'numerical'}, 'L3_S29_F333

In [54]:
from sdv.single_table import TVAESynthesizer

synthesizer = TVAESynthesizer(metadata, epochs=50, verbose=True)
print("\nTVAE modeli eğitiliyor...")
synthesizer.fit(df_response_1_for_gan)
print("TVAE modeli eğitimi tamamlandı.")


TVAE modeli eğitiliyor...


Loss: -840.458: 100%|██████████| 50/50 [03:52<00:00,  4.64s/it]

TVAE modeli eğitimi tamamlandı.


In [55]:
synthesizer.get_loss_values()

,Epoch,Batch,Loss
0,0,0,1566.419800
1,0,1,1405.862427
2,0,2,1267.902100
3,0,3,1163.343384
4,0,4,1171.971680
...,...,...,...
695,49,9,-867.080872
696,49,10,-841.423706
697,49,11,-879.344788
698,49,12,-854.436829


In [56]:
synthetic_data = synthesizer.sample(num_rows=51000)
synthetic_data['Response'] = 1

print("TVAE ile üretilen sentetik veri boyutu:", synthetic_data.shape)

TVAE ile üretilen sentetik veri boyutu: (51000, 411)


In [57]:
synthetic_data.shape

(51000, 411)

In [58]:
synthetic_data.head()

,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,L0_S0_F18,...,L3_S36_D3932,L3_S36_D3936,L3_S36_D3940,L3_S37_D3942,L3_S37_D3943,L3_S37_D3945,L3_S37_D3947,L3_S37_D3949,L3_S37_D3951,Response
0,0.002104,0.018295,-0.179199,-0.179810,0.030792,0.117493,-0.014900,-0.071472,-0.060120,0.005260,...,0.441162,0.450195,0.444336,0.321045,0.398193,0.411865,0.349854,0.375732,0.316162,1
1,0.003870,0.021957,-0.179565,-0.182129,0.031128,0.114075,-0.015198,-0.071655,-0.060638,0.006531,...,0.445801,0.446777,0.445068,0.676270,0.720215,0.792969,0.742676,0.755859,0.726074,1
2,0.002424,0.019928,-0.179199,-0.179932,0.031708,0.116638,-0.015213,-0.072266,-0.059662,0.009232,...,0.444336,0.444336,0.448242,0.353760,0.338379,0.411377,0.331787,0.357910,0.450928,1
3,0.003122,0.018707,-0.176514,-0.182983,0.030640,0.115784,-0.015869,-0.071960,-0.063416,0.007843,...,0.448730,0.444092,0.446533,0.440186,0.429199,0.443115,0.443359,0.441895,0.431396,1
4,0.002163,0.017456,-0.006294,-0.179077,0.030853,0.116638,-0.015274,-0.071472,-0.061249,0.007702,...,0.443604,0.446777,0.447754,0.564941,0.557129,0.526367,0.567383,0.471680,0.540527,1


### GaussianCopulaSynthesizer

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

synthesizer = GaussianCopulaSynthesizer(metadata)
print("\nGaussianCopulaSynthesizer modeli eğitiliyor...")
synthesizer.fit(df_response_1_for_gan)
print("GaussianCopulaSynthesizer modeli eğitimi tamamlandı.")

# synthetic_data = synthesizer.sample(num_rows=10)


GaussianCopulaSynthesizer modeli eğitiliyor...


In [ ]:
synthesizer.get_learned_distributions()

In [ ]:
synthetic_data = synthesizer.sample(num_rows=5000)
synthetic_data['Response'] = 1

print("GaussianCopulaSynthesizer ile üretilen sentetik veri boyutu:", synthetic_data.shape)

# Anomaly Detection

In [34]:
import pandas as pd

train_paths = {
    'categorical': '/kaggle/input/bosch-production-line-performance/train_categorical.csv.zip',
    'date': '/kaggle/input/bosch-production-line-performance/train_date.csv.zip',
    'numeric': '/kaggle/input/bosch-production-line-performance/train_numeric.csv.zip'
}

# 1️⃣ İlk olarak df_processed'deki sütunları al
target_columns = df_processed.columns.tolist()

# 2️⃣ Response = 0 olanlardan 100k örnek çekmek için fonksiyon
def get_response_0_data(paths_dict, target_columns, max_samples=100000, chunksize=10000):
    dfs = {k: [] for k in paths_dict.keys()}
    total_collected = 0

    zip_readers = {k: pd.read_csv(v, compression='zip', chunksize=chunksize) for k, v in paths_dict.items()}

    while total_collected < max_samples:
        try:
            chunks = {k: next(reader) for k, reader in zip_readers.items()}
        except StopIteration:
            break  # dosya bitti

        # numeric chunk içinde Response varsa, filtrele
        if 'Response' in chunks['numeric'].columns:
            mask = (chunks['numeric']['Response'] == 0)
        else:
            continue

        filtered_chunks = {k: chunk[mask] for k, chunk in chunks.items()}

        # stop if nothing left
        if len(filtered_chunks['numeric']) == 0:
            continue

        for k in dfs:
            dfs[k].append(filtered_chunks[k])

        total_collected += len(filtered_chunks['numeric'])
        if total_collected >= max_samples:
            break

    # 3️⃣ Concatenate and merge
    cat_df = pd.concat(dfs['categorical'], axis=0).reset_index(drop=True)
    date_df = pd.concat(dfs['date'], axis=0).reset_index(drop=True)
    num_df = pd.concat(dfs['numeric'], axis=0).reset_index(drop=True)

    # Id bazlı merge
    merged_df = num_df.merge(cat_df, on='Id', how='left')
    merged_df = merged_df.merge(date_df, on='Id', how='left')

    # Sadece hedef sütunları al (kesin tutarlı olsun diye)
    final_df = merged_df[[col for col in target_columns if col in merged_df.columns]]

    print(f"✅ Response=0 alınan veri şekli: {final_df.shape}")
    return final_df


In [35]:
df_response_0 = get_response_0_data(train_paths, df_processed.columns.tolist(), max_samples=100000)

# Ardından df_processed ile birleştir (içinde 1'ler ve sentetik olanlar var)
df_final = pd.concat([df_processed, df_response_0], axis=0).reset_index(drop=True)
print("📦 Final veri seti:", df_final.shape)
print("🔍 Response dağılımı:\n", df_final['Response'].value_counts())

✅ Response=0 alınan veri şekli: (109373, 412)
📦 Final veri seti: (116252, 412)
🔍 Response dağılımı:
 Response
0    109373
1      6879
Name: count, dtype: int64


In [36]:
df_processed = robust_impute_and_encode(df_final)
print("Final shape:", df_processed.shape)
print("Kalan NaN:", df_processed.isnull().sum().sum())  # 0 olmalı

Final shape: (116252, 412)
Kalan NaN: 0


In [58]:
synthetic_data = synthetic_data.copy()

# ID ataması (df_processed'in kaldığı yerden devam)
synthetic_data['Id'] = range(df_processed['Id'].max() + 1, df_processed['Id'].max() + 1 + len(synthetic_data))

# Label ekle (bunlar sentetik olduğu için Response=1)
# synthetic_data['Response'] = 1
synthetic_data.head()

,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,L0_S0_F18,...,L3_S36_D3936,L3_S36_D3940,L3_S37_D3942,L3_S37_D3943,L3_S37_D3945,L3_S37_D3947,L3_S37_D3949,L3_S37_D3951,Response,Id
0,0.047943,0.072144,-0.256836,-0.259766,0.067566,-0.043793,-0.024979,-0.128540,-0.109619,0.017899,...,0.490967,0.490967,0.741211,0.741211,0.740723,0.741211,0.741211,0.740723,1,2366693
1,-0.068420,-0.060333,-0.127319,-0.119324,0.039856,-0.100769,0.002108,-0.026794,-0.012810,-0.031281,...,0.289307,0.289307,0.162354,0.162476,0.162354,0.162354,0.162476,0.162354,1,2366694
2,-0.017868,0.033447,-0.213257,-0.223145,0.018860,0.089539,-0.010063,-0.069092,-0.064697,0.043274,...,0.264893,0.264893,0.394287,0.394287,0.394287,0.394287,0.394043,0.394287,1,2366695
3,-0.196899,-0.195801,-0.185791,-0.196533,-0.001263,-0.166870,0.027695,0.132446,-0.094299,-0.099609,...,0.203369,0.203247,0.100586,0.100647,0.100586,0.100769,0.100647,0.100586,1,2366696
4,-0.047211,-0.002356,-0.150879,-0.135864,-0.031494,0.141479,0.008881,-0.029953,0.235107,0.124084,...,0.479980,0.479980,0.401855,0.401855,0.402100,0.402100,0.402100,0.402100,1,2366697


In [59]:
df_merged = pd.concat([df_processed, synthetic_data], axis=0).reset_index(drop=True)

print("Yeni veri şekli:", df_merged.shape)
print("Yeni Response dağılımı:\n", df_merged['Response'].value_counts())

Yeni veri şekli: (230625, 412)
Yeni Response dağılımı:
 Response
0    218746
1     11879
Name: count, dtype: int64


In [60]:
df_merged.head()

,Id,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,...,L3_S36_D3928,L3_S36_D3932,L3_S36_D3936,L3_S36_D3940,L3_S37_D3942,L3_S37_D3943,L3_S37_D3945,L3_S37_D3947,L3_S37_D3949,L3_S37_D3951
0,1053,0.003000,0.018997,-0.178955,-0.178955,0.031006,0.116028,-0.014999,-0.072021,-0.061005,...,5.944085e-08,5.944085e-08,5.944085e-08,5.944085e-08,5.944016e-08,5.944016e-08,5.944016e-08,5.944016e-08,5.944016e-08,5.944016e-08
1,1250,0.075012,0.101013,-0.178955,-0.215942,-0.013000,0.070007,-0.022003,-0.151978,0.086975,...,1.226019e-07,1.226019e-07,1.226019e-07,1.226019e-07,1.226005e-07,1.226005e-07,1.226005e-07,1.226005e-07,1.226005e-07,1.226005e-07
2,1350,0.068970,0.040985,0.330078,0.330078,-0.099976,-0.293945,0.008003,0.088013,-0.091980,...,1.288023e-07,1.288023e-07,1.288023e-07,1.288023e-07,1.288008e-07,1.288008e-07,1.288008e-07,1.288008e-07,1.288008e-07,1.288008e-07
3,1793,0.003000,-0.026001,0.330078,0.293945,0.073975,0.161011,0.022003,0.128052,-0.198975,...,2.275121e-07,2.275121e-07,2.275121e-07,2.275121e-07,2.275095e-07,2.275095e-07,2.275095e-07,2.275095e-07,2.275095e-07,2.275095e-07
4,2347,-0.114014,-0.161011,0.330078,0.330078,-0.013000,0.116028,0.045013,0.288086,0.036011,...,1.510409e-07,1.510409e-07,1.510409e-07,1.510409e-07,1.509565e-07,1.509565e-07,1.509565e-07,1.509565e-07,1.509565e-07,1.509565e-07


In [42]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df_processed.copy()
cat_cols = df_encoded.select_dtypes(include=['object', 'category']).columns.tolist()

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    le_dict[col] = le  # gerekiyorsa daha sonra inverse yaparsın

In [38]:
from sklearn.preprocessing import StandardScaler

# Id ve Response dışındaki tüm sütunlar
X_cols = [col for col in df_encoded.columns if col not in ['Id', 'Response']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded[X_cols])

# DataFrame'e dönüştür
X_scaled_df = pd.DataFrame(X_scaled, columns=X_cols)
X_scaled_df['Response'] = df_encoded['Response'].values
X_scaled_df['Id'] = df_encoded['Id'].values

In [39]:
X = X_scaled_df.drop(columns=['Id', 'Response'])
y = X_scaled_df['Response']

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [72]:
from sklearn.experimental import enable_iterative_imputer # Deneysel bir özellik olduğu için bu satır gerekli
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor # Kullanılacak tahminci modeli
imputer_iterative = IterativeImputer(
    estimator=RandomForestRegressor(random_state=42), # Veya LGBMRegressor
    max_iter=10, # İterasyon sayısı
    random_state=42,
    verbose=2 # İlerleme çubuğu veya çıktı için
)
X_train_imputed = imputer_iterative.fit_transform(X_train)
X_test_imputed = imputer_iterative.transform(X_test)

[IterativeImputer] Completing matrix with shape (180500, 410)


KeyboardInterrupt: 

In [41]:
from sklearn.impute import KNNImputer

imputer_knn = KNNImputer(n_neighbors=5) # n_neighbors varsayılan olarak 5'tir.
X_train_imputed = imputer_knn.fit_transform(X_train)
X_test_imputed = imputer_knn.transform(X_test)

In [65]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')  # veya median
X_train_imputed = imputer.fit_transform(X_train)

In [66]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')  # veya median
X_train_imputed = imputer.fit_transform(X_train)

In [67]:
X_test_imputed = imputer.transform(X_test)

In [65]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)

In [66]:
X_test_scaled = scaler.transform(X_test_imputed)

## Isolation Forest 

In [43]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=1000,
    contamination='auto',
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
iso.fit(X_train_imputed)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    2.7s remaining:    2.7s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    2.7s finished


IsolationForest(n_estimators=1000, n_jobs=-1, random_state=42, verbose=1)

In [44]:
# Predict: -1 → anomali, 1 → normal
y_pred_iso = iso.predict(X_test_imputed)

# Tahminleri 0/1 haline getir
y_pred_iso = (y_pred_iso == -1).astype(int)

In [45]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, matthews_corrcoef

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_iso))
print("\nClassification Report:\n", classification_report(y_test, y_pred_iso))
print("F1 Score:", f1_score(y_test, y_pred_iso))
print("MCC:", matthews_corrcoef(y_test, y_pred_iso))

Confusion Matrix:
 [[17679  4196]
 [    0  1376]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.81      0.89     21875
           1       0.25      1.00      0.40      1376

    accuracy                           0.82     23251
   macro avg       0.62      0.90      0.65     23251
weighted avg       0.96      0.82      0.86     23251

F1 Score: 0.3960852043753598
MCC: 0.44674374459646116


In [78]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, matthews_corrcoef
import numpy as np
import pandas as pd

def prepare_and_run_isolation_forest(df, id_col='Id', label_col='Response', contamination=0.01):

    df = df.copy()

    # 1. ID kaldır (varsa)
    if id_col in df.columns:
        df.drop(columns=[id_col], inplace=True)

    # 2. Kategorik encode
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # 3. Eksik değerleri doldur (sayısal ve diğerler için basit yöntem)
    imputer = KNNImputer(n_neighbors=5) # n_neighbors varsayılan olarak 5'tir.
    df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

    # 4. Scale işlemi (sadece response=0 ile fit)
    scaler = StandardScaler()
    X_train = df_imputed[df[label_col] == 0].drop(columns=[label_col])
    X_test = df_imputed.drop(columns=[label_col])
    y_test = df[label_col].values

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Eğitim (response=0) verisi shape: {X_train_scaled.shape}")

    # 5. Isolation Forest modeli
    iso = IsolationForest(
        n_estimators=500,
        contamination=contamination,
        max_samples='auto',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    iso.fit(X_train_scaled)

    # 6. Test tahmini
    y_pred = iso.predict(X_test_scaled)
    # -1 anomali, 1 normal → binary etikete dönüştür
    y_pred_bin = (y_pred == -1).astype(int)

    # 7. Performans metriği
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_bin))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_bin, digits=4))

    print(f"F1 Score: {f1_score(y_test, y_pred_bin):.4f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred_bin):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, y_pred_bin):.4f}")

    return iso, scaler, imputer

# Kullanımı:
iso_model, scaler, imputer = prepare_and_run_isolation_forest(df_final)

KeyboardInterrupt: 

In [70]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, matthews_corrcoef
import numpy as np
import pandas as pd

def prepare_and_run_isolation_forest(df, id_col='Id', label_col='Response', contamination=0.01):

    df = df.copy()

    # 1. ID kaldır (varsa)
    if id_col in df.columns:
        df.drop(columns=[id_col], inplace=True)

    # 2. Kategorik encode
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # 3. Eksik değerleri doldur (sayısal ve diğerler için basit yöntem)
    imputer = SimpleImputer(strategy='mean')
    df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

    # 4. Scale işlemi (sadece response=0 ile fit)
    scaler = StandardScaler()
    X_train = df_imputed[df[label_col] == 0].drop(columns=[label_col])
    X_test = df_imputed.drop(columns=[label_col])
    y_test = df[label_col].values

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Eğitim (response=0) verisi shape: {X_train_scaled.shape}")

    # 5. Isolation Forest modeli
    iso = IsolationForest(
        n_estimators=500,
        contamination=contamination,
        max_samples='auto',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    iso.fit(X_train_scaled)

    # 6. Test tahmini
    y_pred = iso.predict(X_test_scaled)
    # -1 anomali, 1 normal → binary etikete dönüştür
    y_pred_bin = (y_pred == -1).astype(int)

    # 7. Performans metriği
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_bin))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_bin, digits=4))

    print(f"F1 Score: {f1_score(y_test, y_pred_bin):.4f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred_bin):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, y_pred_bin):.4f}")

    return iso, scaler, imputer

# Kullanımı:
iso_model, scaler, imputer = prepare_and_run_isolation_forest(df_final)

Eğitim (response=0) verisi shape: (109373, 410)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    1.4s remaining:    1.4s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    1.5s finished



Confusion Matrix:
[[108279   1094]
 [  1120   5759]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9898    0.9900    0.9899    109373
           1     0.8404    0.8372    0.8388      6879

    accuracy                         0.9810    116252
   macro avg     0.9151    0.9136    0.9143    116252
weighted avg     0.9809    0.9810    0.9809    116252

F1 Score: 0.8388
MCC: 0.8287
ROC AUC: 0.9136


In [1]:
from sklearn.svm import OneClassSVM

def prepare_and_run_one_class_svm(df, id_col='Id', label_col='Response', kernel='rbf', nu=0.01, gamma='scale'):

    df = df.copy()

    # 1. ID kaldır (varsa)
    if id_col in df.columns:
        df.drop(columns=[id_col], inplace=True)

    # 2. Kategorik encode
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # 3. Eksik değerleri doldur (basit yöntem)
    imputer = SimpleImputer(strategy='mean')
    df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

    # 4. Scale işlemi (sadece response=0 ile fit)
    scaler = StandardScaler()
    X_train = df_imputed[df[label_col] == 0].drop(columns=[label_col])
    X_test = df_imputed.drop(columns=[label_col])
    y_test = df[label_col].values

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Eğitim (response=0) verisi shape: {X_train_scaled.shape}")

    # 5. One-Class SVM modeli
    oc_svm = OneClassSVM(
        kernel=kernel,
        nu=nu,
        gamma=gamma
    )

    oc_svm.fit(X_train_scaled)

    # 6. Test tahmini
    y_pred = oc_svm.predict(X_test_scaled)
    # 1 normal, -1 anomali, binary etikete dönüştür
    y_pred_bin = (y_pred == -1).astype(int)

    # 7. Performans metriği
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_bin))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_bin, digits=4))

    print(f"F1 Score: {f1_score(y_test, y_pred_bin):.4f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred_bin):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, y_pred_bin):.4f}")

    return oc_svm, scaler, imputer

# Kullanımı:
ocsvm_model, scaler, imputer = prepare_and_run_one_class_svm(df_final)


NameError: name 'df_final' is not defined

In [72]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, matthews_corrcoef
import pandas as pd
import numpy as np

def prepare_and_run_lof(df, id_col='Id', label_col='Response', n_neighbors=20, contamination=0.01):
    df = df.copy()

    # 1. ID kaldır
    if id_col in df.columns:
        df.drop(columns=[id_col], inplace=True)

    # 2. Kategorik encode
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # 3. Eksik değerleri doldur
    imputer = SimpleImputer(strategy='mean')
    df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

    # 4. Scale işlemi (fit sadece response=0 ile)
    scaler = StandardScaler()
    X_train = df_imputed[df[label_col] == 0].drop(columns=[label_col])
    X_test = df_imputed.drop(columns=[label_col])
    y_test = df[label_col].values

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Eğitim (response=0) verisi shape: {X_train_scaled.shape}")

    # 5. LOF Modeli (novelty=True test için)
    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        novelty=True  # novelty=True ile fit-test ayrımı yapılır
    )
    lof.fit(X_train_scaled)

    # 6. Tahmin
    y_pred = lof.predict(X_test_scaled)
    # 1 normal, -1 anomali → binary etiket
    y_pred_bin = (y_pred == -1).astype(int)

    # 7. Performans
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_bin))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_bin, digits=4))

    print(f"F1 Score: {f1_score(y_test, y_pred_bin):.4f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred_bin):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, y_pred_bin):.4f}")

    return lof, scaler, imputer

# Kullanımı:
lof_model, scaler, imputer = prepare_and_run_lof(df_final)


Confusion Matrix:
[[108363   1010]
 [     0   6879]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9908    0.9954    109373
           1     0.8720    1.0000    0.9316      6879

    accuracy                         0.9913    116252
   macro avg     0.9360    0.9954    0.9635    116252
weighted avg     0.9924    0.9913    0.9916    116252

F1 Score: 0.9316
MCC: 0.9295
ROC AUC: 0.9954


In [48]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer # KNNImputer'ı import ettiğinizden emin olun
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, matthews_corrcoef
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

def prepare_and_run_autoencoder_anomaly(df, id_col='Id', label_col='Response', encoding_dim=None,
                                        epochs=50, batch_size=32, validation_split=0.2, patience=5,
                                        threshold_quantile=0.95):
    """
    Veriyi ön işler ve Autoencoder tabanlı anomali sınıflandırması yapar.

    Args:
        df (pd.DataFrame): Orijinal DataFrame.
        id_col (str): ID sütununun adı.
        label_col (str): Etiket (response) sütununun adı. Anomali tespiti için 0 (normal) ve 1 (anomali) beklenir.
        encoding_dim (int): Autoencoder'ın gizli katmanının boyutu. Varsayılan None ise,
                            giriş boyutunun yarısı olarak ayarlanır.
        epochs (int): Autoencoder eğitimi için epoch sayısı.
        batch_size (int): Eğitim batch boyutu.
        validation_split (float): Eğitim verisinin yüzde kaçının doğrulama için ayrılacağı.
        patience (int): Early Stopping için sabır değeri.
        threshold_quantile (float): Anomali skoru (rekonstrüksiyon hatası) için eşik belirlemede kullanılacak niceklik.
                                    Örn: 0.95, hatanın en büyük %5'ini anomali kabul eder.

    Returns:
        tuple: (autoencoder_model, encoder_model, decoder_model, scaler, imputer, history)
               - autoencoder_model: Eğitilmiş Autoencoder modeli.
               - encoder_model: Autoencoder'ın encoder kısmı.
               - decoder_model: Autoencoder'ın decoder kısmı.
               - scaler: Veriyi ölçeklemek için kullanılan StandardScaler objesi.
               - imputer: Eksik değerleri doldurmak için kullanılan KNNImputer objesi.
               - history: Model eğitim geçmişi.
    """

    df_copy = df.copy()

    # 1. ID sütununu kaldır (varsa)
    if id_col in df_copy.columns:
        print(f"'{id_col}' sütunu kaldırılıyor.")
        df_copy.drop(columns=[id_col], inplace=True)

    # Label sütununu ayır
    y_true = df_copy[label_col]
    X_data = df_copy.drop(columns=[label_col])

    # 2. Kategorik sütunları Label Encode et
    print("Kategorik sütunlar Label Encode ediliyor.")
    cat_cols = X_data.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        # NaN değerleri LabelEncoder'ın işleyebilmesi için string'e çevir ve doldur
        X_data[col] = X_data[col].astype(str).replace('None', 'unknown').fillna("unknown")
        le = LabelEncoder()
        X_data[col] = le.fit_transform(X_data[col])

    # 3. Eksik değerleri KNNImputer ile doldur
    # KNNImputer'dan önce sayısal sütunlar da LabelEncode edildiği için tüm sütunları işler.
    print("Eksik değerler KNNImputer ile dolduruluyor.")
    # imputer = KNNImputer(n_neighbors=5)
    imputer = SimpleImputer(strategy='median')
    # fit_transform, NumPy array döndürür, DataFrame'e geri çevir
    X_imputed = pd.DataFrame(imputer.fit_transform(X_data), columns=X_data.columns)

    # 4. Veriyi ölçekle (sadece Response=0 (Normal) verisi ile fit et)
    # Anomali tespiti için Autoencoder'ı yalnızca normal veri üzerinde eğitmek esastır.
    print("Veri StandardScaler ile ölçekleniyor (sadece Normal sınıf üzerinde fit edilecek).")
    X_normal_train = X_imputed[y_true == 0] # Sadece normal veriyi al
    
    scaler = StandardScaler()
    X_normal_train_scaled = scaler.fit_transform(X_normal_train) # Sadece normal veri ile scaler'ı eğit
    
    # Tüm veri setini (normal + anomali) aynı scaler ile dönüştür
    X_all_scaled = scaler.transform(X_imputed)

    print(f"Autoencoder eğitimi için kullanılan normal verinin boyutu: {X_normal_train_scaled.shape}")

    # 5. Autoencoder Modelini Oluştur
    input_dim = X_normal_train_scaled.shape[1]
    
    # Eğer encoding_dim belirtilmediyse, giriş boyutunun yarısını kullan
    if encoding_dim is None:
        encoding_dim = input_dim // 2
        if encoding_dim == 0: # Ensure it's at least 1 if input_dim is 1
            encoding_dim = 1 

    print(f"Autoencoder mimarisi: Input({input_dim}) -> Dense({encoding_dim}) -> Dense({input_dim})")

    # Encoder
    input_layer = Input(shape=(input_dim,))
    encoder = Dense(encoding_dim, activation="relu")(input_layer)

    # Decoder
    decoder = Dense(input_dim, activation="linear")(encoder) # Çıkış katmanı için doğrusal aktivasyon genellikle daha iyidir

    # Autoencoder Modeli
    autoencoder = Model(inputs=input_layer, outputs=decoder)
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse') # MSE (Mean Squared Error) yaygın kayıp fonksiyonudur

    # Early Stopping callback
    early_stopping = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    print("\nAutoencoder modeli eğitiliyor...")
    history = autoencoder.fit(
        X_normal_train_scaled, X_normal_train_scaled, # X_train = Y_train çünkü autoencoder kendi çıktısını öğrenir
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=[early_stopping],
        verbose=1 # Eğitim ilerlemesini göster
    )
    print("Autoencoder modeli eğitimi tamamlandı.")

    # 6. Rekonstrüksiyon Hatalarını Hesapla
    # Tüm veri seti üzerinde (normal ve anomali) yeniden yapılandırma hatasını hesapla
    X_reconstructed = autoencoder.predict(X_all_scaled)
    reconstruction_errors = np.mean(np.square(X_all_scaled - X_reconstructed), axis=1)

    print(f"Rekonstrüksiyon hataları hesaplandı. Boyut: {reconstruction_errors.shape}")

    # 7. Anomali Eşiğini Belirle
    # Eşik, normal verinin rekonstrüksiyon hatalarının dağılımına göre belirlenir.
    # Genellikle %95 veya %99'luk niceklik kullanılır.
    threshold = np.quantile(reconstruction_errors[y_true == 0], threshold_quantile)
    print(f"Anomali eşiği ({threshold_quantile*100:.0f}. niceklik): {threshold:.4f}")

    # 8. Anomali Sınıflandırması Yap
    # Rekonstrüksiyon hatası eşiği aşan noktaları anomali (-1 veya 1) olarak işaretle
    y_pred_anomaly = (reconstruction_errors > threshold).astype(int)

    # 9. Performans Metriklerini Hesapla
    print("\n--- Anomali Sınıflandırma Performansı ---")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred_anomaly))

    print("\nClassification Report:")
    # target_names'ı doğru sınıflar için güncelleyebilirsiniz (0: Normal, 1: Anomali)
    print(classification_report(y_true, y_pred_anomaly, target_names=['Normal (0)', 'Anomaly (1)'], digits=4))

    print(f"F1 Score (Anomali Sınıfı için): {f1_score(y_true, y_pred_anomaly):.4f}")
    print(f"MCC: {matthews_corrcoef(y_true, y_pred_anomaly):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_true, y_pred_anomaly):.4f}")

    # İsteğe bağlı: Encoder ve Decoder'ı ayrı ayrı döndür
    encoder_model = Model(inputs=input_layer, outputs=encoder)
    decoder_input = Input(shape=(encoding_dim,))
    decoder_model = Model(inputs=decoder_input, outputs=autoencoder.layers[-1](decoder_input)) # Son katman decoder katmanı

    return autoencoder, encoder_model, decoder_model, scaler, imputer, history, y_pred_anomaly, reconstruction_errors, threshold


# Eğer Autoencoder'ı çalıştırmak istiyorsanız, bu fonksiyonu çağırın:
autoencoder_model, encoder, decoder, scaler, imputer, history, y_pred_ae, ae_errors, ae_threshold = \
     prepare_and_run_autoencoder_anomaly(df=df_final, id_col='Id', label_col='Response',
                                         encoding_dim=None, # Varsayılan olarak giriş boyutunun yarısı
                                         epochs=100,       # Daha fazla epoch ile deneyin
                                         batch_size=64,
                                         validation_split=0.15,
                                         patience=10,      # Erken durma sabrı
                                         threshold_quantile=0.97) # Anomali eşiği nicekliği

# # Sonuçları incelemek için:
# # print(ae_errors[:10]) # İlk 10 hatayı gör
# # print(y_pred_ae[:10]) # İlk 10 tahmini gör
# # print(ae_threshold) # Belirlenen eşiği gör

'Id' sütunu kaldırılıyor.
Kategorik sütunlar Label Encode ediliyor.
Eksik değerler KNNImputer ile dolduruluyor.
Veri StandardScaler ile ölçekleniyor (sadece Normal sınıf üzerinde fit edilecek).
Autoencoder eğitimi için kullanılan normal verinin boyutu: (109373, 410)
Autoencoder mimarisi: Input(410) -> Dense(205) -> Dense(410)

Autoencoder modeli eğitiliyor...
Epoch 1/100
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1976 - val_loss: 0.0722
Epoch 2/100
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0322 - val_loss: 0.0480
Epoch 3/100
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0354 - val_loss: 0.0389
Epoch 4/100
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0258 - val_loss: 0.0590
Epoch 5/100
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0299 - val_loss: 0.0649
Epoch 6/100
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0241 - val_loss: 0.0385
Epoch 7/100
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0366 - val_loss: 0.0362
Epoch 8/100
145

In [51]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, matthews_corrcoef
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

def prepare_and_run_vae_anomaly(df, id_col='Id', label_col='Response', latent_dim=10,
                                encoding_layer_dim=64, epochs=50, batch_size=32,
                                validation_split=0.2, patience=5, threshold_quantile=0.95):
    """
    Veriyi ön işler ve Variational Autoencoder (VAE) tabanlı anomali sınıflandırması yapar.

    Args:
        df (pd.DataFrame): Orijinal DataFrame.
        id_col (str): ID sütununun adı.
        label_col (str): Etiket (response) sütununun adı. Anomali tespiti için 0 (normal) ve 1 (anomali) beklenir.
        latent_dim (int): VAE'nin latent uzayının boyutu.
        encoding_layer_dim (int): Encoder ve Decoder'ın ara katmanının boyutu.
        epochs (int): VAE eğitimi için epoch sayısı.
        batch_size (int): Eğitim batch boyutu.
        validation_split (float): Eğitim verisinin yüzde kaçının doğrulama için ayrılacağı.
        patience (int): Early Stopping için sabır değeri.
        threshold_quantile (float): Anomali skoru (rekonstrüksiyon hatası) için eşik belirlemede kullanılacak niceklik.

    Returns:
        tuple: (vae_model, encoder_model, decoder_model, scaler, imputer, history)
               - vae_model: Eğitilmiş VAE modeli.
               - encoder_model: VAE'nin encoder kısmı.
               - decoder_model: VAE'nin decoder kısmı.
               - scaler: Veriyi ölçeklemek için kullanılan StandardScaler objesi.
               - imputer: Eksik değerleri doldurmak için kullanılan KNNImputer objesi.
               - history: Model eğitim geçmişi.
               - y_pred_vae: VAE'nin anomali tahminleri.
               - vae_errors: Her bir veri noktasının rekonstrüksiyon hatası.
               - vae_threshold: Belirlenen anomali eşiği.
    """

    df_copy = df.copy()

    # 1. ID sütununu kaldır (varsa)
    if id_col in df_copy.columns:
        print(f"'{id_col}' sütunu kaldırılıyor.")
        df_copy.drop(columns=[id_col], inplace=True)

    # Label sütununu ayır
    y_true = df_copy[label_col]
    X_data = df_copy.drop(columns=[label_col])

    # 2. Kategorik sütunları Label Encode et
    print("Kategorik sütunlar Label Encode ediliyor.")
    cat_cols = X_data.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        X_data[col] = X_data[col].astype(str).replace('None', 'unknown').fillna("unknown")
        le = LabelEncoder()
        X_data[col] = le.fit_transform(X_data[col])

    # 3. Eksik değerleri KNNImputer ile doldur
    print("Eksik değerler KNNImputer ile dolduruluyor.")
    imputer = SimpleImputer(strategy='median')

    X_imputed = pd.DataFrame(imputer.fit_transform(X_data), columns=X_data.columns)

    # 4. Veriyi ölçekle (sadece Response=0 (Normal) verisi ile fit et)
    print("Veri StandardScaler ile ölçekleniyor (sadece Normal sınıf üzerinde fit edilecek).")
    X_normal_train = X_imputed[y_true == 0]
    
    scaler = StandardScaler()
    X_normal_train_scaled = scaler.fit_transform(X_normal_train)
    
    X_all_scaled = scaler.transform(X_imputed)

    print(f"VAE eğitimi için kullanılan normal verinin boyutu: {X_normal_train_scaled.shape}")

    # --- VAE Modelini Oluştur ---
    input_dim = X_normal_train_scaled.shape[1]

    # Encoder Mimarisi
    inputs = Input(shape=(input_dim,), name='encoder_input')
    x = Dense(encoding_layer_dim, activation='relu', name='encoder_dense_1')(inputs)
    z_mean = Dense(latent_dim, name='z_mean')(x)
    z_log_var = Dense(latent_dim, name='z_log_var')(x)

    # Yeniden Parametrelendirme Hilesi
    def sampling(args):
        z_mean, z_log_var = args
        batch = K.shape(z_mean)[0]
        dim = K.int_shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + K.exp(0.5 * z_log_var) * epsilon

    z = Lambda(sampling, output_shape=(latent_dim,), name='z_sampling')([z_mean, z_log_var])

    # Encoder Modeli
    encoder = Model(inputs, [z_mean, z_log_var, z], name='encoder')
    encoder.summary()

    # Decoder Mimarisi
    latent_inputs = Input(shape=(latent_dim,), name='z_sampling_input')
    decoder_x = Dense(encoding_layer_dim, activation='relu', name='decoder_dense_1')(latent_inputs)
    outputs = Dense(input_dim, activation='linear', name='decoder_output')(decoder_x) # Çıkış katmanı doğrusal

    # Decoder Modeli
    decoder = Model(latent_inputs, outputs, name='decoder')
    decoder.summary()

    # VAE Modeli (Encoder ve Decoder'ı birleştir)
    vae_outputs = decoder(encoder(inputs)[2]) # encoder'ın z (sampled latent vector) çıktısını kullan
    vae = Model(inputs, vae_outputs, name='vae')

    # VAE Kayıp Fonksiyonu (Rekonstrüksiyon Kaybı + KL Divergence Kaybı)
    reconstruction_loss = tf.reduce_mean(
        tf.reduce_sum(tf.keras.losses.mse(inputs, vae_outputs), axis=-1)
    ) # Mean Squared Error (MSE)
    
    kl_loss = -0.5 * tf.reduce_mean(
        tf.reduce_sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=1)
    )
    vae_loss = reconstruction_loss + kl_loss
    vae.add_loss(vae_loss) # Modelin özel kayıp fonksiyonunu ekle

    vae.compile(optimizer=Adam(learning_rate=0.001))
    vae.summary()

    # Early Stopping callback
    early_stopping = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    print("\nVAE modeli eğitiliyor...")
    history = vae.fit(
        X_normal_train_scaled, # Giriş verisi
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=[early_stopping],
        verbose=1
    )
    print("VAE modeli eğitimi tamamlandı.")

    # 6. Rekonstrüksiyon Hatalarını Hesapla
    X_reconstructed = vae.predict(X_all_scaled)
    reconstruction_errors = np.mean(np.square(X_all_scaled - X_reconstructed), axis=1)

    print(f"Rekonstrüksiyon hataları hesaplandı. Boyut: {reconstruction_errors.shape}")

    # 7. Anomali Eşiğini Belirle
    threshold = np.quantile(reconstruction_errors[y_true == 0], threshold_quantile)
    print(f"Anomali eşiği ({threshold_quantile*100:.0f}. niceklik): {threshold:.4f}")

    # 8. Anomali Sınıflandırması Yap
    y_pred_anomaly = (reconstruction_errors > threshold).astype(int)

    # 9. Performans Metriklerini Hesapla
    print("\n--- Anomali Sınıflandırma Performansı ---")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred_anomaly))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_anomaly, target_names=['Normal (0)', 'Anomaly (1)'], digits=4))

    print(f"F1 Score (Anomali Sınıfı için): {f1_score(y_true, y_pred_anomaly):.4f}")
    print(f"MCC: {matthews_corrcoef(y_true, y_pred_anomaly):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_true, y_pred_anomaly):.4f}")

    return vae, encoder, decoder, scaler, imputer, history, y_pred_anomaly, reconstruction_errors, threshold

# --- Kullanım Örneği ---
# df_final, tüm verinizin olduğu DataFrame olmalı (ID ve Response dahil)
# df_final = pd.read_csv("your_data.csv") # Örnek veri yükleme

vae_model, vae_encoder, vae_decoder, scaler_vae, imputer_vae, history_vae, y_pred_vae, vae_errors, vae_threshold = \
     prepare_and_run_vae_anomaly(df=df_final, id_col='Id', label_col='Response',
                                 latent_dim=20,           # Latent uzay boyutu
                                 encoding_layer_dim=128,  # Encoder/Decoder ara katman boyutu
                                 epochs=100,
                                 batch_size=64,
                                 validation_split=0.15,
                                 patience=15,             # Daha fazla sabır değeri
                                 threshold_quantile=0.97) # Anomali eşiği nicekliği

# # Sonuçları incelemek için:
# # print(vae_errors[:10]) # İlk 10 hatayı gör
# # print(y_pred_vae[:10]) # İlk 10 tahmini gör
# # print(vae_threshold) # Belirlenen eşiği gör

'Id' sütunu kaldırılıyor.
Kategorik sütunlar Label Encode ediliyor.
Eksik değerler KNNImputer ile dolduruluyor.
Veri StandardScaler ile ölçekleniyor (sadece Normal sınıf üzerinde fit edilecek).
VAE eğitimi için kullanılan normal verinin boyutu: (109373, 410)


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_input             │ (None, 410)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ encoder_dense_1 (Dense)   │ (None, 128)            │         52,608 │ encoder_input[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ z_mean (Dense)            │ (None, 20)             │          2,580 │ encoder_dense_1[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ z_log_var (Dense)         │ (None, 20)             │          2,580 │ encoder_dense_1[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ z_sampling (Lambda)       │ (None, 20)             │              0 │ z_mean[0][0],          │
│                           │                        │                │ z_log_var[0][0]        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 57,768 (225.66 KB)

 Trainable params: 57,768 (225.66 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ z_sampling_input (InputLayer)        │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ decoder_dense_1 (Dense)              │ (None, 128)                 │           2,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ decoder_output (Dense)               │ (None, 410)                 │          52,890 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 55,578 (217.10 KB)

 Trainable params: 55,578 (217.10 KB)

 Non-trainable params: 0 (0.00 B)

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.operations`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, matthews_corrcoef
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

def prepare_and_run_vae_ml_ensemble(df, id_col='Id', label_col='Response', latent_dim=10,
                                   encoding_layer_dim=64, epochs=50, batch_size=32,
                                   validation_split=0.2, patience=5, threshold_quantile=0.95,
                                   test_size=0.2, random_state=42):
    """
    VAE + ML Ensemble yaklaşımıyla anomali tespiti yapar.
    
    Args:
        df (pd.DataFrame): Orijinal DataFrame.
        id_col (str): ID sütununun adı.
        label_col (str): Etiket (response) sütununun adı.
        latent_dim (int): VAE'nin latent uzayının boyutu.
        encoding_layer_dim (int): Encoder ve Decoder'ın ara katmanının boyutu.
        epochs (int): VAE eğitimi için epoch sayısı.
        batch_size (int): Eğitim batch boyutu.
        validation_split (float): VAE eğitiminde validation split oranı.
        patience (int): Early Stopping için sabır değeri.
        threshold_quantile (float): VAE anomali skoru için eşik nicekliği.
        test_size (float): ML modelleri için test split oranı.
        random_state (int): Rastgelelik kontrolü için seed.
    
    Returns:
        dict: Tüm modellerin sonuçlarını içeren sözlük.
    """
    
    print("="*80)
    print("VAE + ML ENSEMBLE ANOMALY DETECTION PIPELINE")
    print("="*80)
    
    df_copy = df.copy()
    
    # 1. Ön İşleme
    print("\n1. VERİ ÖN İŞLEME")
    print("-" * 40)
    
    if id_col in df_copy.columns:
        print(f"'{id_col}' sütunu kaldırılıyor.")
        df_copy.drop(columns=[id_col], inplace=True)
    
    y_true = df_copy[label_col]
    X_data = df_copy.drop(columns=[label_col])
    
    print(f"Veri boyutu: {X_data.shape}")
    print(f"Sınıf dağılımı:\n{y_true.value_counts()}")
    
    # Kategorik sütunları encode et
    print("\nKategorik sütunlar Label Encode ediliyor...")
    cat_cols = X_data.select_dtypes(include=['object', 'category']).columns
    label_encoders = {}
    
    for col in cat_cols:
        X_data[col] = X_data[col].astype(str).replace('None', 'unknown').fillna("unknown")
        le = LabelEncoder()
        X_data[col] = le.fit_transform(X_data[col])
        label_encoders[col] = le
    
    # Eksik değerleri doldur
    print("Eksik değerler KNNImputer ile dolduruluyor...")
    imputer = KNNImputer(n_neighbors=5)
    X_imputed = pd.DataFrame(imputer.fit_transform(X_data), columns=X_data.columns)
    
    # Veriyi ölçekle
    print("Veri StandardScaler ile ölçekleniyor...")
    X_normal_train = X_imputed[y_true == 0]
    
    scaler = StandardScaler()
    X_normal_train_scaled = scaler.fit_transform(X_normal_train)
    X_all_scaled = scaler.transform(X_imputed)
    
    # 2. VAE MODELİ
    print("\n2. VAE MODELİ OLUŞTURMA VE EĞİTİM")
    print("-" * 40)
    
    input_dim = X_normal_train_scaled.shape[1]
    print(f"Giriş boyutu: {input_dim}")
    print(f"VAE eğitimi için kullanılan normal veri boyutu: {X_normal_train_scaled.shape}")
    
    # VAE mimarisi
    inputs = Input(shape=(input_dim,), name='encoder_input')
    x = Dense(encoding_layer_dim, activation='relu', name='encoder_dense_1')(inputs)
    z_mean = Dense(latent_dim, name='z_mean')(x)
    z_log_var = Dense(latent_dim, name='z_log_var')(x)
    
    def sampling(args):
        z_mean, z_log_var = args
        batch = K.shape(z_mean)[0]
        dim = K.int_shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + K.exp(0.5 * z_log_var) * epsilon
    
    z = Lambda(sampling, output_shape=(latent_dim,), name='z_sampling')([z_mean, z_log_var])
    
    # Encoder
    encoder = Model(inputs, [z_mean, z_log_var, z], name='encoder')
    
    # Decoder
    latent_inputs = Input(shape=(latent_dim,), name='z_sampling_input')
    decoder_x = Dense(encoding_layer_dim, activation='relu', name='decoder_dense_1')(latent_inputs)
    outputs = Dense(input_dim, activation='linear', name='decoder_output')(decoder_x)
    decoder = Model(latent_inputs, outputs, name='decoder')
    
    # VAE
    vae_outputs = decoder(encoder(inputs)[2])
    vae = Model(inputs, vae_outputs, name='vae')
    
    # Loss function
    reconstruction_loss = tf.reduce_mean(
        tf.reduce_sum(tf.keras.losses.mse(inputs, vae_outputs), axis=-1)
    )
    kl_loss = -0.5 * tf.reduce_mean(
        tf.reduce_sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=1)
    )
    vae_loss = reconstruction_loss + kl_loss
    vae.add_loss(vae_loss)
    vae.compile(optimizer=Adam(learning_rate=0.001))
    
    # VAE eğitimi
    early_stopping = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
    
    print("VAE modeli eğitiliyor...")
    history = vae.fit(
        X_normal_train_scaled,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=[early_stopping],
        verbose=1
    )
    
    # 3. VAE TABANLI ÖZELLİK ÇIKARIMI
    print("\n3. VAE TABANLІ ÖZELLİK ÇIKARIMI")
    print("-" * 40)
    
    # Rekonstrüksiyon hatalarını hesapla
    X_reconstructed = vae.predict(X_all_scaled, verbose=0)
    reconstruction_errors = np.mean(np.square(X_all_scaled - X_reconstructed), axis=1)
    
    # Latent özelliklerini çıkar
    latent_features = encoder.predict(X_all_scaled, verbose=0)[0]  # z_mean kullan
    
    # VAE anomali eşiğini belirle
    vae_threshold = np.quantile(reconstruction_errors[y_true == 0], threshold_quantile)
    y_pred_vae = (reconstruction_errors > vae_threshold).astype(int)
    
    print(f"VAE anomali eşiği ({threshold_quantile*100:.0f}. niceklik): {vae_threshold:.4f}")
    
    # 4. ÖZELLİK SETLERİNİ HAZIRLA
    print("\n4. ÖZELLİK SETLERİNİ HAZIRLA")
    print("-" * 40)
    
    # Farklı özellik kombinasyonları
    feature_sets = {
        'original': X_all_scaled,
        'reconstruction_error': reconstruction_errors.reshape(-1, 1),
        'latent_features': latent_features,
        'combined_vae': np.column_stack([reconstruction_errors, latent_features]),
        'all_features': np.column_stack([X_all_scaled, reconstruction_errors, latent_features])
    }
    
    print("Özellik setleri:")
    for name, features in feature_sets.items():
        print(f"  {name}: {features.shape}")
    
    # 5. MAKINE ÖĞRENMESİ MODELLERİ
    print("\n5. MAKINE ÖĞRENMESİ MODELLERİ")
    print("-" * 40)
    
    # Model tanımları
    models = {
        'Random Forest': RandomForestClassifier(
            n_estimators=100, 
            max_depth=10,
            random_state=random_state,
            n_jobs=-1
        ),
        'Decision Tree': DecisionTreeClassifier(
            max_depth=10,
            random_state=random_state
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=random_state,
            eval_metric='logloss'
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=random_state,
            verbose=-1
        )
    }
    
    # Sonuçları saklamak için
    results = {
        'vae_standalone': {
            'y_pred': y_pred_vae,
            'reconstruction_errors': reconstruction_errors,
            'threshold': vae_threshold
        },
        'ml_results': {}
    }
    
    # Her özellik seti için her model test et
    for feature_name, X_features in feature_sets.items():
        print(f"\n--- {feature_name.upper()} ÖZELLİK SETİ ---")
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_features, y_true, test_size=test_size, 
            random_state=random_state, stratify=y_true
        )
        
        results['ml_results'][feature_name] = {}
        
        for model_name, model in models.items():
            print(f"\n{model_name} eğitiliyor...")
            
            # Modeli eğit
            model.fit(X_train, y_train)
            
            # Tahmin yap
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            
            # Metrikleri hesapla
            f1 = f1_score(y_test, y_pred)
            mcc = matthews_corrcoef(y_test, y_pred)
            roc_auc = roc_auc_score(y_test, y_pred_proba)
            
            # Sonuçları kaydet
            results['ml_results'][feature_name][model_name] = {
                'model': model,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba,
                'y_test': y_test,
                'f1_score': f1,
                'mcc': mcc,
                'roc_auc': roc_auc,
                'confusion_matrix': confusion_matrix(y_test, y_pred)
            }
            
            print(f"  F1 Score: {f1:.4f}")
            print(f"  MCC: {mcc:.4f}")
            print(f"  ROC AUC: {roc_auc:.4f}")
    
    # 6. SONUÇLARI KARŞILAŞTIR
    print("\n6. SONUÇLARIN KARŞILAŞTIRILMASI")
    print("-" * 40)
    
    # VAE standalone performansı
    print("\nVAE Standalone Performansı:")
    print(f"F1 Score: {f1_score(y_true, y_pred_vae):.4f}")
    print(f"MCC: {matthews_corrcoef(y_true, y_pred_vae):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_true, y_pred_vae):.4f}")
    
    # En iyi performansları bul
    best_results = []
    
    for feature_name in feature_sets.keys():
        for model_name in models.keys():
            result = results['ml_results'][feature_name][model_name]
            best_results.append({
                'Feature Set': feature_name,
                'Model': model_name,
                'F1 Score': result['f1_score'],
                'MCC': result['mcc'],
                'ROC AUC': result['roc_auc']
            })
    
    # DataFrame olarak sonuçları göster
    results_df = pd.DataFrame(best_results)
    
    print("\nTüm Model Performansları:")
    print(results_df.round(4))
    
    # En iyi modelleri bul
    best_f1 = results_df.loc[results_df['F1 Score'].idxmax()]
    best_mcc = results_df.loc[results_df['MCC'].idxmax()]
    best_roc = results_df.loc[results_df['ROC AUC'].idxmax()]
    
    print(f"\nEn İyi F1 Score: {best_f1['Model']} ({best_f1['Feature Set']}) - {best_f1['F1 Score']:.4f}")
    print(f"En İyi MCC: {best_mcc['Model']} ({best_mcc['Feature Set']}) - {best_mcc['MCC']:.4f}")
    print(f"En İyi ROC AUC: {best_roc['Model']} ({best_roc['Feature Set']}) - {best_roc['ROC AUC']:.4f}")
    
    # Diğer nesneleri de ekle
    results.update({
        'vae_model': vae,
        'encoder': encoder,
        'decoder': decoder,
        'scaler': scaler,
        'imputer': imputer,
        'label_encoders': label_encoders,
        'history': history,
        'results_df': results_df,
        'feature_sets': feature_sets,
        'best_models': {
            'f1': best_f1,
            'mcc': best_mcc,
            'roc_auc': best_roc
        }
    })
    
    return results

def plot_results(results):
    """
    Sonuçları görselleştir.
    """
    # Performans karşılaştırması
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    df = results['results_df']
    
    # F1 Score
    pivot_f1 = df.pivot(index='Feature Set', columns='Model', values='F1 Score')
    sns.heatmap(pivot_f1, annot=True, fmt='.3f', cmap='Blues', ax=axes[0])
    axes[0].set_title('F1 Score Comparison')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('Feature Set')
    
    # MCC
    pivot_mcc = df.pivot(index='Feature Set', columns='Model', values='MCC')
    sns.heatmap(pivot_mcc, annot=True, fmt='.3f', cmap='Greens', ax=axes[1])
    axes[1].set_title('MCC Comparison')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('Feature Set')
    
    # ROC AUC
    pivot_roc = df.pivot(index='Feature Set', columns='Model', values='ROC AUC')
    sns.heatmap(pivot_roc, annot=True, fmt='.3f', cmap='Reds', ax=axes[2])
    axes[2].set_title('ROC AUC Comparison')
    axes[2].set_xlabel('Model')
    axes[2].set_ylabel('Feature Set')
    
    plt.tight_layout()
    plt.show()
    
    # VAE eğitim geçmişi
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(results['history'].history['loss'], label='Training Loss')
    plt.plot(results['history'].history['val_loss'], label='Validation Loss')
    plt.title('VAE Training History')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Rekonstrüksiyon hatası dağılımı
    plt.subplot(1, 2, 2)
    errors = results['vae_standalone']['reconstruction_errors']
    plt.hist(errors, bins=50, alpha=0.7, label='Reconstruction Errors')
    plt.axvline(results['vae_standalone']['threshold'], color='red', 
                linestyle='--', label=f'Threshold: {results["vae_standalone"]["threshold"]:.4f}')
    plt.title('Reconstruction Error Distribution')
    plt.xlabel('Reconstruction Error')
    plt.ylabel('Frequency')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Kullanım örneği:
# results = prepare_and_run_vae_ml_ensemble(
#     df=df_final, 
#     id_col='Id', 
#     label_col='Response',
#     latent_dim=20,
#     encoding_layer_dim=128,
#     epochs=100,
#     batch_size=64,
#     validation_split=0.15,
#     patience=15,
#     threshold_quantile=0.97,
#     test_size=0.2,
#     random_state=42
# )

# plot_results(results)

In [ ]:
daha hızlı machine learning 
# https://rapids.ai/
# https://github.com/rapidsai/notebooks

Synthetic Data Vault (SDV) 
veri üretimi için sdv kütüphanesi
# https://docs.sdv.dev/sdv